In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import stable_retro
import matplotlib.pyplot as plt
import numpy as np
import gymnasium as gym

ModuleNotFoundError: No module named 'torch'

#### Mario Environment and Rewards

In [12]:
def compute_reward(prev_info, curr_info, done):
    reward = 0.0

    # --- Progress ---
    delta_x = curr_info['x'] - prev_info['x']
    if 0 < delta_x < 200:           # filter out screen wraps
        reward += delta_x * 0.01    # small continuous signal for moving right

    # --- Milestones ---
    if curr_info.get('midpoint_flag', 0) > prev_info.get('midpoint_flag', 0):
        reward += 5.0
    if curr_info.get('finish_flag', 0) > prev_info.get('finish_flag', 0):
        reward += 10.0

    # --- Powerups ---
    if curr_info.get('powerup_status', 0) > prev_info.get('powerup_status', 0):
        reward += 1.0

    # --- Penalties ---
    if curr_info['lives'] < prev_info['lives']:
        reward -= 5.0

    return reward

In [ ]:
class SMWEnv(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        # Override the observation space to match your tuple state
        self.observation_space = gym.spaces.Box(
            low=np.array([0, 0, 0, 0, 0], dtype=np.float32),
            high=np.array([255, 9999, 3, 1, 1], dtype=np.float32),
            dtype=np.float32
        )
        self.prev_info = {}

    def reset(self, **kwargs):
        _, info = self.env.reset(**kwargs)
        self.prev_info = info
        obs = self.get_obs(info)
        return obs, info

    def step(self, action):
        _, _, terminated, truncated, info = self.env.step(action)
        obs = self.get_obs(info)
        reward = compute_reward(self.prev_info, info, terminated or truncated)
        self.prev_info = info
        return obs, reward, terminated, truncated, info

    def get_obs(self, info):
        return np.array([
            info.get('screen', 0),
            info.get('x', 0),
            info.get('powerup_status', 0),
            info.get('riding_yoshi', 0),
            info.get('midpoint_flag', 0),
        ], dtype=np.float32)

#### Actor and Critic

In [ ]:
class Actor(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.fc1 = nn.Linear(obs_dim, 64)   # 5 inputs → 64 neurons
        self.fc2 = nn.Linear(64, 64)        # 64 → 64
        self.fc3 = nn.Linear(64, action_dim) # 64 → one output per action

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        probs = F.softmax(self.fc3(x), dim=-1)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action, log_prob

class Critic(nn.Module):
    def __init__(self, obs_dim):
        super().__init__()
        self.obs_dim = obs_dim
        self.fc1 = nn.Linear(obs_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        V = self.fc3(x)
        return V

#### PPO Algorithm

In [ ]:
class SMW_PPO:
    def __init__(self, env, obs_dim, action_dim, theta, n_workers, epsilon = 0.2, alpha = 0.95, gamma=0.99, num_steps=2048):
        self.env = env
        self.gamma = gamma
        self.alpha = alpha
        self.epsilon = epsilon
        self.num_steps = num_steps
        self.n_workers = n_workers

        self.states    = []
        self.actions   = []
        self.rewards   = []
        self.log_probs = []
        self.values    = []
        self.dones     = []
        self.theta = theta

        self.state, _ = self.env.reset()
        self.actor = Actor(obs_dim=5, action_dim=self.env.action_space.n)
        self.critic = Critic(obs_dim=5)
        self.actor_optimizer  = optim.Adam(self.actor.parameters(),  lr=3e-4)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=3e-4)

        self.policy = []
        
    def collect_trajectories(self):
        for _ in range(self.num_steps):
            action, log_prob = self.actor(self.state)
            value = self.critic(self.state)

            next_state, reward, terminated, truncated, info = self.env.step(action)
            done = terminated or truncated

            self.states.append(self.state)
            self.actions.append(action)
            self.rewards.append(reward)
            self.log_probs.append(log_prob)
            self.values.append(value)
            self.dones.append(done)

            # reset if episode ended
            if done:
                self.state, _ = self.env.reset()
            else:
                self.state = next_state

        return self.states, self.actions, self.rewards, self.log_probs, self.values, self.dones

    def r_to_go(self):
        Rt = []
        discounted_sum = 0
        
        for reward, done in zip(reversed(self.rewards), reversed(self.dones)):
            if done:
                discounted_sum = 0

            discounted_sum = reward + self.gamma * discounted_sum
            Rt.insert(0, discounted_sum)

        return Rt

    # def advantage_est(self):
    #     gae = torch.GAE(self.n_workers, self.num_steps, self.gamma, self.alpha)
    #     return gae

    def advantage_est(self):
        Rt = torch.tensor(self.r_to_go(), dtype=torch.float32)
        V  = torch.stack(self.values).squeeze()
        advantages = Rt - V.detach()
        return advantages
    
    def update(self):
        states_t = torch.tensor(np.array(self.states),    dtype=torch.float32)
        
        # get new log_probs and values from current networks
        _, new_log_probs = self.actor(states_t)
        new_values = self.critic(states_t).squeeze()

        # compute the probability ratio π_new / π_old
        old_logs = torch.stack(self.log_probs).detach()
        ratio = torch.exp(new_log_probs - old_logs)

        # compute advantages
        advantages = self.advantage_est().detach()

        #two candidates for the loss
        unclipped = ratio * advantages
        clipped = torch.clamp(ratio, 1-self.epsilon, 1+self.epsilon) * advantages

        # take the minimum (pessimistic bound)
        actor_loss = -torch.mean(torch.min(unclipped, clipped))
        
        # critic just minimizes prediction error
        critic_loss = nn.MSELoss()(new_values, torch.tensor(self.r_to_go(), dtype=torch.float32))

        # backprop actor
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # backprop critic
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
    def rollout():
        

SyntaxError: expected ':' (1666871249.py, line 65)

In [14]:
returns_sum = {}
returns_count = {}

num_eps = 50

base_env = stable_retro.make('SuperMarioWorld-Snes-v0', render_mode=None)
env = SMWEnv(base_env)

# tracking
total_rewards = []
steps_per_ep = []
max_x_per_ep = []
v_size_per_ep = []

# plotting
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('SMW PPO Training', fontsize=14)

eps = range(1, num_eps + 1)

axes[0, 0].plot(eps, total_rewards)
axes[0, 0].set_title('Total Reward per Episode')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')

axes[0, 1].plot(eps, steps_per_ep)
axes[0, 1].set_title('Steps per Episode')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Steps')

axes[1, 0].plot(eps, max_x_per_ep)
axes[1, 0].set_title('Max X Reached per Episode')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('X Position')

axes[1, 1].plot(eps, v_size_per_ep)
axes[1, 1].set_title('Value Function Size (# States)')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('# States in V')

plt.tight_layout()
plt.savefig('mc_results.png')
plt.show()

RuntimeError: Cannot create multiple emulator instances per process, make sure to call env.close() on each environment before creating a new one

In [28]:
# test a bunch of addresses around where x should be
import stable_retro

env = stable_retro.make(game='SuperMarioWorld-Snes-v0', render_mode='human')
env.reset()

for _ in range(100):
    action = [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]  # hold right
    env.step(action)

ram = env.get_ram()
# print addresses in the range we care about
for addr in [0x00D1, 0x00D3, 0x0086, 0x00D4, 0x00B6, 0x00B8]:
    print(f"0x{addr:04X}: {ram[addr]}")
    print(0x7E0000 + 0x00D1)  # x
    print(0x7E0000 + 0x00D4)  # screen  

env.close()

0x00D1: 129
8257745
8257748
0x00D3: 96
8257745
8257748
0x0086: 0
8257745
8257748
0x00D4: 1
8257745
8257748
0x00B6: 0
8257745
8257748
0x00B8: 0
8257745
8257748
